# Network checks — example Kuramoto network

End-to-end sanity check of a single graph built from a connectivity matrix.

This notebook:

- Loads the Kuramoto connectome adjacency shipped with the repo
- Builds a `networkx` graph and computes Forman-Ricci curvature
- Summarises the network with `hyphi.analyses.summarize_network`
- Plots the network, colours edges by curvature, and compares the curvature distribution before/after weight pruning

Uses repo-relative paths so it runs without editing on any machine.

## Notebook settings

In [ ]:
verbose = True

## Imports

In [ ]:
from pathlib import Path

import networkx as nx
import numpy as np

from hyphi.analyses import (
    prune_graph_by_weight,
    remove_self_loops_copy,
    summarize_network,
)
from hyphi.io import load_network_pkl
from hyphi.modeling.graph_curvatures import compute_frc, extract_curvatures
from hyphi.visualization.network_plots import (
    plot_curvature_distribution,
    plot_curvature_network,
    plot_curvature_network_layouts,
    plot_network,
    plot_weight_distribution,
)

## Small example: FRC on a made-up adjacency matrix

In [ ]:
A = np.array([
    [0, 1, 1, 0],
    [1, 0, 1, 0],
    [1, 1, 0, 1],
    [0, 0, 1, 0],
], dtype=float)

G_toy = nx.from_numpy_array(A)
G_toy_frc = compute_frc(G_toy, method="1d")
curvatures_toy = extract_curvatures(G_toy_frc, curvature="formanCurvature")

if verbose:
    summarize_network(G_toy, show_n=3, title="Summary of toy network")
    print("\nToy curvatures:", curvatures_toy)

## Load the Kuramoto adjacency matrix

The pickle shipped in `data/connectome/` holds a single 152×152 adjacency matrix.
Adjust `REPO_ROOT` if you run this notebook from a different working directory.

In [ ]:
REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent

pickle_path = REPO_ROOT / "data" / "connectome" / "1_connectome_kuramoto.pkl"
loaded = load_network_pkl(str(pickle_path))

if isinstance(loaded, np.ndarray):
    adjacency = loaded
elif isinstance(loaded, nx.Graph):
    adjacency = nx.to_numpy_array(loaded)
else:
    raise TypeError(f"Unexpected pickle contents: {type(loaded)!r}")

if verbose:
    print("adjacency type:", type(adjacency).__name__)
    print("adjacency shape:", adjacency.shape)

## Construct the network and compute FRC

In [ ]:
G = nx.from_numpy_array(adjacency)
G_frc = compute_frc(G, method="1d")
curvatures = extract_curvatures(G_frc, curvature="formanCurvature")

if verbose:
    summarize_network(G, show_n=10, title="Summary of Kuramoto network")

## Visualise the network

Self-loops distort layouts, so drop them for visualisation only.

In [ ]:
H = remove_self_loops_copy(G_frc)
plot_weight_distribution(
    H,
    bins=20,
    title="Distribution of edge weights (self-loops removed)",
);

### Pruned network

In [ ]:
threshold = 0.5
H_pruned = prune_graph_by_weight(H, threshold=threshold, keep_all_nodes=False)

if verbose:
    summarize_network(
        H_pruned,
        show_n=10,
        title=f"Summary of pruned Kuramoto network (weight >= {threshold})",
    )

plot_network(
    H_pruned,
    layout="spring",
    title=f"Pruned Kuramoto network (weight >= {threshold})",
    node_size=500,
    font_size=8,
);

### Edges coloured by curvature

In [ ]:
plot_curvature_network(
    H_pruned,
    curvature_attr="formanCurvature",
    layout="spring",
    title=f"Pruned Kuramoto network coloured by Forman-Ricci curvature (weight >= {threshold})",
    node_size=100,
    font_size=8,
    width=2,
);

### Same pruned network across multiple layouts

Useful for checking whether high-curvature edges localise structurally rather than as layout artefacts.

In [ ]:
fig = plot_curvature_network_layouts(
    H_pruned,
    curvature_attr="formanCurvature",
    layouts=("spring", "kamada_kawai", "circular"),
    figsize=(18, 6),
    with_labels=False,
    node_size=80,
    node_alpha=0.35,
    font_size=8,
    width=2,
    suptitle=(
        f"Pruned Kuramoto network coloured by Forman-Ricci curvature "
        f"(edge weight threshold >= {threshold})"
    ),
);

## Distribution of Forman-Ricci curvature

Compare the curvature distribution of the full graph (dominated by self-loop contributions in the unpruned view) against the pruned subgraph.

In [ ]:
plot_curvature_distribution(
    curvatures,
    bins=20,
    title="Distribution of Forman-Ricci curvature",
);

curvatures_pruned = extract_curvatures(H_pruned, curvature="formanCurvature")
plot_curvature_distribution(
    curvatures_pruned,
    bins=20,
    title=f"Distribution of Forman-Ricci curvature (pruned, weight >= {threshold})",
);